# MIDA507 -- Notebook Session 05
## Qualite des Donnees : Framework DAMA et Nettoyage Python

| | |
|---|---|
| **Session** | S05 -- Qualite des donnees et nettoyage professionnel |
| **Prerequis** | Notebooks S01-S04 |
| **Duree** | 70 a 90 minutes |

### Objectifs
1. Comprendre les 6 dimensions de qualite du framework DAMA
2. Profiler automatiquement un jeu de donnees selon ces 6 dimensions
3. Appliquer des regles de nettoyage Python professionnelles
4. Produire un rapport de qualite structure
5. Documenter les corrections dans le journal de lineage

### Livrables : `MIDA507_S05_rapport_qualite.json` + jeu nettoyé


---
## PARTIE 1 -- Configuration


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, json, os, random, hashlib
from datetime import datetime, timedelta
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
PAL={"ter1":"#3E1500","ter2":"#BF360C","ter3":"#E64A19",
     "olive":"#558B2F","bleu":"#1565C0","gris":"#546E7A","clair":"#FBE9E7"}
def regen(n=5000):
    random.seed(42); np.random.seed(42)
    g=["These et memoire","Manuel universitaire","Revue scientifique","Rapport de recherche","Ouvrage de reference","Document administratif"]
    fi=["Droit","Sciences eco.","Lettres modernes","Histoire","Geographie","Medecine","Agronomie","Informatique"]
    ni=["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
    re=["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
    la=["Francais","Anglais","Bilingue FR/EN","Arabe","Autres lang. africaines"]
    li=["fra","eng","fra+eng","ara","mul"]
    d0=datetime(2018,1,1)
    df=pd.DataFrame({
        "ark_id":[f"ark:/67375/UNG-{str(i+1).zfill(6)}" for i in range(n)],
        "cote":[f"UNG-{str(i+1).zfill(5)}" for i in range(n)],
        "genre_document":np.random.choice(g,n,p=[0.28,0.30,0.15,0.10,0.10,0.07]),
        "date_emprunt":[(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(n)],
        "duree_jours":np.random.choice([3,7,14,21,30],n,p=[0.1,0.3,0.3,0.2,0.1]),
        "code_usager_anon":[f"USR-{hex(random.randint(0,0xFFFFFF))[2:].upper().zfill(6)}" for _ in range(n)],
        "filiere":np.random.choice(fi,n),
        "niveau":np.random.choice(ni,n),
        "region_origine":np.random.choice(re,n),
        "langue_document":np.random.choice(la,n,p=[0.62,0.22,0.10,0.04,0.02]),
        "langue_iso639":np.random.choice(li,n,p=[0.62,0.22,0.10,0.04,0.02]),
    })
    df["annee"]=pd.to_datetime(df["date_emprunt"]).dt.year
    # Introduire des problemes de qualite realistes
    idx1=np.random.choice(df.index,size=int(n*0.08),replace=False)
    df.loc[idx1,"filiere"]="non renseign"  # valeur invalide
    idx2=np.random.choice(df.index,size=int(n*0.03),replace=False)
    df.loc[idx2,"duree_jours"]=999          # valeur aberrante
    idx3=np.random.choice(df.index,size=int(n*0.04),replace=False)
    df.loc[idx3,"langue_document"]=None     # valeur manquante
    return df
CSV="MIDA507_S02_bibliotheque_ung_enrichie.csv"
if os.path.exists(CSV):
    df=pd.read_csv(CSV); print(f"Jeu S02 charge : {len(df):,} lignes")
else:
    df=regen(); print(f"Jeu regenere avec problemes de qualite : {len(df):,} lignes")
print(f"Colonnes : {list(df.columns)}")


---
## PARTIE 2 -- Les 6 Dimensions DAMA de la Qualite des Donnees

> Le DAMA (Data Management Association) definit 6 dimensions fondamentales
> pour evaluer la qualite d'un jeu de donnees. Chaque dimension est mesurable
> avec Python et conduit a des actions correctives specifiques.


In [ ]:
DIMENSIONS_DAMA = {
    "Completude": {
        "definition": "Proportion de champs renseignes sur le total attendu",
        "calcul_python": "1 - df.isnull().mean()",
        "seuil_acceptable": 0.95,
        "action_IDA": "Imputer ou etiqueter les valeurs manquantes (ex: 'Non renseigne')"
    },
    "Coherence": {
        "definition": "Cohérence des valeurs entre les champs (ex: date retour > date emprunt)",
        "calcul_python": "regles metier definies par l'IDA",
        "seuil_acceptable": 0.99,
        "action_IDA": "Definir les regles metier dans le DMP, alerter sur les violations"
    },
    "Exactitude": {
        "definition": "Precision des valeurs par rapport a la realite (ex: durees plausibles)",
        "calcul_python": "df.between() ou comparaison avec referentiels",
        "seuil_acceptable": 0.97,
        "action_IDA": "Definir les plages valides, corriger les valeurs aberrantes"
    },
    "Unicite": {
        "definition": "Absence de doublons (un meme evenement n'est enregistre qu'une fois)",
        "calcul_python": "df.duplicated().sum()",
        "seuil_acceptable": 1.0,
        "action_IDA": "Identifier la cle primaire, supprimer ou fusionner les doublons"
    },
    "Validite": {
        "definition": "Respect des domaines de valeurs autorises (ex: genres documentaires valides)",
        "calcul_python": "df['col'].isin(valeurs_autorisees).mean()",
        "seuil_acceptable": 0.95,
        "action_IDA": "Definir et documenter le dictionnaire de donnees avec les domaines"
    },
    "Actualite": {
        "definition": "Fraicheur des donnees (dernier enregistrement pas trop ancien)",
        "calcul_python": "(today - df['date'].max()).days",
        "seuil_acceptable": 365,  # jours
        "action_IDA": "Planifier la frequence de MAJ dans le DMP, alerter si depasse"
    }
}

# Affichage synthétique
print("FRAMEWORK DAMA -- 6 DIMENSIONS DE QUALITE")
print("="*65)
for dim, info in DIMENSIONS_DAMA.items():
    print(f"\n  {dim.upper()}")
    print(f"    Definition : {info['definition']}")
    print(f"    Seuil      : {info['seuil_acceptable']}")
    print(f"    Action IDA : {info['action_IDA']}")
print()
print("Prochaine etape : calculer chaque dimension sur notre jeu BU-UNG.")


---
## PARTIE 3 -- Profil de Qualite Automatique du Jeu BU-UNG


In [ ]:
from datetime import date

def profiler_qualite(df, nom_jeu="Jeu de donnees"):
    """Calcule les 6 dimensions DAMA et retourne un rapport structure."""
    rapport = {
        "jeu": nom_jeu,
        "date_profil": date.today().isoformat(),
        "nb_lignes": len(df),
        "nb_colonnes": len(df.columns),
        "dimensions": {}
    }

    # 1. COMPLETUDE
    completude_par_col = (1 - df.isnull().mean()).round(4)
    completude_globale = completude_par_col.mean()
    rapport["dimensions"]["completude"] = {
        "score": round(float(completude_globale), 4),
        "detail_par_colonne": completude_par_col.to_dict(),
        "colonnes_problematiques": completude_par_col[completude_par_col < 0.95].index.tolist(),
        "conforme": completude_globale >= 0.95
    }

    # 2. UNICITE
    nb_doublons = df.duplicated().sum()
    rapport["dimensions"]["unicite"] = {
        "score": round(1 - nb_doublons/len(df), 4),
        "nb_doublons": int(nb_doublons),
        "conforme": nb_doublons == 0
    }

    # 3. EXACTITUDE (duree_jours)
    if "duree_jours" in df.columns:
        valides = df["duree_jours"].between(1, 90)
        score_ex = valides.mean()
        aberrants = df[~valides][["duree_jours"]].head(5).to_dict("records")
    else:
        score_ex = 1.0; aberrants = []
    rapport["dimensions"]["exactitude"] = {
        "score": round(float(score_ex), 4),
        "nb_aberrants_duree": int((~df["duree_jours"].between(1,90)).sum()) if "duree_jours" in df.columns else 0,
        "exemples_aberrants": aberrants,
        "conforme": score_ex >= 0.97
    }

    # 4. VALIDITE (genres documentaires)
    GENRES_VALIDES = {"These et memoire","Manuel universitaire","Revue scientifique",
                      "Rapport de recherche","Ouvrage de reference","Document administratif",
                      "Atlas et cartographie","Periodique"}
    if "genre_document" in df.columns:
        valides_genre = df["genre_document"].isin(GENRES_VALIDES)
        score_val = valides_genre.mean()
        invalides = df[~valides_genre]["genre_document"].value_counts().head(5).to_dict()
    else:
        score_val = 1.0; invalides = {}
    rapport["dimensions"]["validite"] = {
        "score": round(float(score_val), 4),
        "nb_valeurs_invalides": int((~df["genre_document"].isin(GENRES_VALIDES)).sum()) if "genre_document" in df.columns else 0,
        "genres_invalides_top5": invalides,
        "conforme": score_val >= 0.95
    }

    # 5. COHERENCE (filiere non vide et valide)
    FILIERES_VALIDES = {"Droit","Sciences eco.","Lettres modernes","Histoire",
                        "Geographie","Medecine","Agronomie","Informatique",
                        "Sciences de l'education","Sociologie","Non renseigne","Non renseignee"}
    if "filiere" in df.columns:
        fil_non_null = df["filiere"].notna()
        fil_valide = df["filiere"].isin(FILIERES_VALIDES)
        score_coh = (fil_non_null & fil_valide).mean()
        inval_fil = df[~fil_valide & fil_non_null]["filiere"].value_counts().head(5).to_dict()
    else:
        score_coh = 1.0; inval_fil = {}
    rapport["dimensions"]["coherence"] = {
        "score": round(float(score_coh), 4),
        "nb_filieres_invalides": int((~df["filiere"].isin(FILIERES_VALIDES)).sum()) if "filiere" in df.columns else 0,
        "filieres_invalides_top5": inval_fil,
        "conforme": score_coh >= 0.99
    }

    # 6. ACTUALITE
    try:
        derniere_date = pd.to_datetime(df["date_emprunt"]).max()
        jours_depuis  = (pd.Timestamp(date.today()) - derniere_date).days
    except:
        jours_depuis = 0
    rapport["dimensions"]["actualite"] = {
        "score": min(1.0, max(0.0, 1 - jours_depuis/730)),
        "jours_depuis_dernier_enregistrement": jours_depuis,
        "conforme": jours_depuis <= 365,
        "note": "Le jeu couvre 2018-2023 -- actualisation annuelle recommandee"
    }

    # Score global
    scores = [v["score"] for v in rapport["dimensions"].values()]
    rapport["score_global"] = round(sum(scores)/len(scores)*100, 1)
    rapport["dimensions_conformes"] = sum(v["conforme"] for v in rapport["dimensions"].values())
    rapport["nb_dimensions"] = len(rapport["dimensions"])

    return rapport

rapport = profiler_qualite(df, "BU-UNG Emprunts 2018-2023")
print(f"RAPPORT DE QUALITE DAMA -- {rapport['jeu']}")
print(f"Date : {rapport['date_profil']} | Lignes : {rapport['nb_lignes']:,}")
print("="*60)
print(f"  Score global : {rapport['score_global']}/100")
print(f"  Dimensions conformes : {rapport['dimensions_conformes']}/{rapport['nb_dimensions']}")
print()
for dim, data in rapport["dimensions"].items():
    conforme = "OK" if data["conforme"] else "A CORRIGER"
    print(f"  {dim.upper():<20} Score : {data['score']:.3f}   [{conforme}]")


---
## PARTIE 4 -- Nettoyage Python Professionnel

> On applique les corrections identifiees par le profil de qualite.
> Chaque correction est documentee pour le journal de lineage.


In [ ]:
def nettoyer_jeu(df_brut):
    """Applique les 5 regles de nettoyage DAMA sur le jeu BU-UNG."""
    df = df_brut.copy()
    log = []

    # REGLE 1 : Supprimer les durees aberrantes (>90 jours ou <1 jour)
    if "duree_jours" in df.columns:
        masque = df["duree_jours"].between(1, 90)
        n_supp = (~masque).sum()
        df = df[masque].copy()
        log.append({"regle":"R1 -- Durees aberrantes",
                    "action":f"Suppression de {n_supp} enregistrements avec duree hors [1,90] jours",
                    "nb_corriges":int(n_supp), "dimension_dama":"Exactitude"})

    # REGLE 2 : Standardiser les valeurs de filiere invalides
    if "filiere" in df.columns:
        FILIERES_VALIDES = {"Droit","Sciences eco.","Lettres modernes","Histoire",
                            "Geographie","Medecine","Agronomie","Informatique"}
        invalides_mask = ~df["filiere"].isin(FILIERES_VALIDES) | df["filiere"].isna()
        n_corr = invalides_mask.sum()
        df.loc[invalides_mask, "filiere"] = "Non renseigne"
        log.append({"regle":"R2 -- Filieres invalides",
                    "action":f"Remplacement de {n_corr} valeurs invalides par 'Non renseigne'",
                    "nb_corriges":int(n_corr), "dimension_dama":"Validite + Completude"})

    # REGLE 3 : Remplir les langues manquantes par 'Inconnu'
    if "langue_document" in df.columns:
        null_lang = df["langue_document"].isna().sum()
        df["langue_document"] = df["langue_document"].fillna("Inconnu")
        if "langue_iso639" in df.columns:
            df.loc[df["langue_iso639"].isna(), "langue_iso639"] = "und"  # ISO 639 undetermined
        log.append({"regle":"R3 -- Langues manquantes",
                    "action":f"Remplacement de {null_lang} valeurs null par 'Inconnu' (ISO: 'und')",
                    "nb_corriges":int(null_lang), "dimension_dama":"Completude"})

    # REGLE 4 : Supprimer les doublons exacts
    n_avant = len(df)
    cols_cle = [c for c in ["cote","date_emprunt","code_usager_anon"] if c in df.columns]
    if cols_cle:
        df = df.drop_duplicates(subset=cols_cle, keep="first")
    n_supprim = n_avant - len(df)
    log.append({"regle":"R4 -- Doublons",
                "action":f"Suppression de {n_supprim} doublons (meme cote+date+usager)",
                "nb_corriges":int(n_supprim), "dimension_dama":"Unicite"})

    # REGLE 5 : Standardiser les noms de colonnes
    df.columns = [c.lower().strip().replace(" ","_") for c in df.columns]
    log.append({"regle":"R5 -- Noms de colonnes",
                "action":"Normalisation : minuscules + underscores (conformite DCAT)",
                "nb_corriges":int(len(df.columns)), "dimension_dama":"Coherence"})

    return df, log

df_net, log_nettoyage = nettoyer_jeu(df)

print("NETTOYAGE APPLIQUE -- JOURNAL DES CORRECTIONS")
print("="*60)
print(f"  Avant nettoyage : {len(df):,} lignes | {len(df.columns)} colonnes")
print(f"  Apres nettoyage : {len(df_net):,} lignes | {len(df_net.columns)} colonnes")
print()
for regle in log_nettoyage:
    print(f"  [{regle['dimension_dama']}] {regle['regle']}")
    print(f"    {regle['action']}")
    print()

# Rapport final de qualite apres nettoyage
rapport_apres = profiler_qualite(df_net, "BU-UNG apres nettoyage S05")
print(f"Score de qualite AVANT : {rapport['score_global']}/100")
print(f"Score de qualite APRES : {rapport_apres['score_global']}/100")
print(f"Gain de qualite        : +{rapport_apres['score_global'] - rapport['score_global']:.1f} points")


In [ ]:
# Visualisation comparative avant/après
dims = list(rapport["dimensions"].keys())
scores_avant = [rapport["dimensions"][d]["score"]*100 for d in dims]
scores_apres = [rapport_apres["dimensions"][d]["score"]*100 for d in dims]

x = np.arange(len(dims))
width = 0.35
fig, ax = plt.subplots(figsize=(13,6))
b1 = ax.bar(x-width/2, scores_avant, width, label=f"Avant (score global: {rapport['score_global']}/100)",
            color=PAL["gris"], alpha=0.8, edgecolor="white")
b2 = ax.bar(x+width/2, scores_apres, width, label=f"Apres (score global: {rapport_apres['score_global']}/100)",
            color=PAL["ter2"], alpha=0.9, edgecolor="white")
ax.axhline(y=95, color="#CC0000", linestyle="--", alpha=0.5, label="Seuil qualite (95%)")
ax.set_xticks(x); ax.set_xticklabels([d.capitalize() for d in dims], rotation=20, ha="right")
ax.set_ylabel("Score (%)"); ax.set_ylim(0, 110)
ax.set_title("Qualite DAMA -- BU-UNG : Avant vs Apres nettoyage S05",
             fontsize=13, fontweight="bold", color=PAL["ter1"])
for bar in b2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{bar.get_height():.0f}%", ha="center", fontsize=9, fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

# Export du rapport et du jeu nettoyé
rapport["log_nettoyage"] = log_nettoyage
with open("MIDA507_S05_rapport_qualite.json","w",encoding="utf-8") as f:
    json.dump(rapport, f, ensure_ascii=False, indent=2)
df_net.to_csv("MIDA507_S05_bibliotheque_ung_nettoyee.csv", index=False, encoding="utf-8-sig")
print("Rapport qualite sauvegarde : MIDA507_S05_rapport_qualite.json")
print(f"Jeu nettoyé sauvegarde    : MIDA507_S05_bibliotheque_ung_nettoyee.csv ({len(df_net):,} lignes)")


---
### EXERCICE -- Profil de qualite sur votre jeu de donnees


In [ ]:
# ============================================================
# EXERCICE -- Profil DAMA de votre jeu reel
# ============================================================
# OPTION A : Chargez votre propre jeu de donnees CSV
# OPTION B : Repondez aux questions ci-dessous sur un jeu de votre institution

print("ANALYSE DE QUALITE -- MON INSTITUTION")
print("="*55)

# Estimations qualitatives (si vous n'avez pas de jeu de donnees sous la main)
MON_JEU = "[Nom et description du jeu de donnees de votre institution]"
MES_ESTIMATIONS = {
    "completude": None,     # <- estimez 0.0 a 1.0 (ex: 0.85 = 85% de champs remplis)
    "unicite": None,        # <- estimez 0.0 a 1.0 (ex: 0.99 = 1% de doublons estimes)
    "exactitude": None,     # <- estimez 0.0 a 1.0
    "validite": None,       # <- estimez 0.0 a 1.0
    "coherence": None,      # <- estimez 0.0 a 1.0
    "actualite": None,      # <- estimez 0.0 a 1.0
}
MON_PROBLEME_PRINCIPAL = "[Decrivez en 1 phrase le principal probleme de qualite identifie]"
MA_SOLUTION_IDA        = "[Quelle regle de nettoyage proposez-vous en priorite ?]"

valides = {k:v for k,v in MES_ESTIMATIONS.items() if isinstance(v,(int,float)) and 0<=v<=1}
if valides:
    score_moyen = sum(valides.values())/len(valides)*100
    print(f"Jeu : {MON_JEU}")
    print()
    for dim, val in MES_ESTIMATIONS.items():
        if val is not None:
            barre = chr(9608)*int(val*20) + chr(9617)*(20-int(val*20))
            print(f"  {dim:<15} : {val:.2f}  {barre}")
    print(f"\n  Score DAMA estime : {score_moyen:.1f}/100")
    print(f"  Probleme principal : {MON_PROBLEME_PRINCIPAL}")
    print(f"  Solution IDA       : {MA_SOLUTION_IDA}")
else:
    print("[A COMPLETER] Entrez des estimations 0.0-1.0 pour chaque dimension.")
    print("Exemple : 'completude': 0.85  (85% des champs sont remplis)")


---
## Bilan S05

| Competence | Statut |
|---|---|
| Connaitre les 6 dimensions DAMA | OK |
| Profiler un jeu de donnees automatiquement | OK |
| Appliquer 5 regles de nettoyage Python | OK |
| Produire un rapport de qualite structure | OK |
| Profiler mon propre jeu de donnees | A completer |

**Lien S06 :** Le jeu nettoyé sera publie sur CKAN avec pipeline complet.

*Notebook MIDA507 S05 -- Master MIDA -- Universite de Douala*
